In [0]:
from pyspark.sql import functions as F
S, G = "workspace.silver", "workspace.gold"

def save(df, name):
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
      .saveAsTable(f"{G}.{name}")
    print(name, df.count())

# ---- dimensions
save(spark.table(f"{S}.customers").select(
    "customer_id", "customer_unique_id", "customer_city", "customer_state"), "dim_customer")

save(spark.table(f"{S}.sellers").select(
    "seller_id", "seller_city", "seller_state"), "dim_seller")

save(spark.table(f"{S}.products").select(
    "product_id", "category", "product_weight_g", "product_photos_qty"), "dim_product")

dim_date = spark.sql("""
  SELECT d AS date, year(d) AS year, quarter(d) AS quarter, month(d) AS month,
         date_format(d, 'MMM') AS month_name, date_format(d, 'yyyy-MM') AS year_month,
         dayofweek(d) AS day_of_week
  FROM (SELECT explode(sequence(DATE'2016-01-01', DATE'2018-12-31', INTERVAL 1 DAY)) AS d)
""")
save(dim_date, "dim_date")

# ---- fact: one row per order item
o = spark.table(f"{S}.orders")
i = spark.table(f"{S}.order_items")

delivered = F.to_date("order_delivered_customer_date")
estimated = F.to_date("order_estimated_delivery_date")

fact = (o.join(i, "order_id")
  .select(
      "order_id", "order_item_id", "customer_id", "product_id", "seller_id",
      F.to_date("order_purchase_timestamp").alias("order_date"),
      "order_status", "price", "freight_value",
      delivered.alias("delivered_date"),
      estimated.alias("estimated_delivery_date"),
      F.when(delivered.isNotNull(), 1).otherwise(0).alias("is_delivered"),
      F.when(delivered.isNotNull() & (delivered <= estimated), 1).otherwise(0).alias("is_on_time"),
  ))
save(fact, "fact_orders")